In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

# Model

## XGB Baseline

In [4]:
SEED = 7

In [5]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [6]:
X = df_train.drop(columns=['id', 'Calories'])
X = pd.get_dummies(X, columns=cat_vars, drop_first=True)
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

In [7]:
xgb_baseline = xgb.XGBRegressor(enable_calegorical=True)
xgb_baseline.fit(X_train, y_train)

y_val_pred = xgb_baseline.predict(X_val)
y_val_pred = np.maximum(y_val_pred, 0)
score = mean_squared_log_error(y_val_pred, y_val)
print(f'XGB Baseline Score: {score}')

XGB Baseline Score: 0.004272607917848008


## Big Tuna

In [8]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [9]:
def ensure_positive(y_pred):
    return np.maximum(y_pred, 0)  # Replace negative values with 0

def msle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = mean_squared_log_error(y_true, y_pred)
    return 'MSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_msle(X, y, params, num_folds=5, num_boost_round=1000, early_stopping_rounds=50):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            evals=[(dval, 'val')],
            feval=msle_eval,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        
        # Get predictions and ensure they're positive
        y_val_pred = model.predict(dval)
        y_val_pred = np.maximum(0, y_val_pred)
        score = mean_squared_log_error(y_val, y_val_pred)
        # print(score)
        fold_scores.append(score)
        
    return fold_scores

In [10]:
def objective(trial):
    params = {
        # "objective": "reg:squarederror",
        # "eval_metric": "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "random_state": SEED
    }
    
    score = np.mean(xgb_cv_msle(X=X, y=y, params=params))
    return score

In [11]:
%%time
study = optuna.create_study(direction='minimize',
                            sampler = optuna.samplers.RandomSampler(seed=SEED),
                            study_name = "BIG BLUE FIN TUNA!!")
study.optimize(objective, n_trials=50, show_progress_bar=True, )

[I 2025-05-03 12:22:14,380] A new study created in memory with name: BIG BLUE FIN TUNA!!


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-05-03 12:44:30,299] Trial 0 finished with value: 0.003913689784030101 and parameters: {'learning_rate': 0.03, 'max_depth': 17, 'subsample': 0.8, 'colsample_bytree': 0.9, 'max_bin': 2009, 'min_child_weight': 6, 'gamma': 2.5056023182996894, 'lambda': 0.0019418001640357841, 'alpha': 0.011851025167261414, 'grow_policy': 'lossguide'}. Best is trial 0 with value: 0.003913689784030101.
[I 2025-05-03 12:46:06,157] Trial 1 finished with value: 0.003907401982973365 and parameters: {'learning_rate': 0.09, 'max_depth': 11, 'subsample': 0.6, 'colsample_bytree': 0.7, 'max_bin': 1886, 'min_child_weight': 3, 'gamma': 2.2606198090884155, 'lambda': 5.3066944048916245, 'alpha': 0.001257757484073345, 'grow_policy': 'lossguide'}. Best is trial 1 with value: 0.003907401982973365.
[I 2025-05-03 12:47:42,659] Trial 2 finished with value: 0.004015433298111664 and parameters: {'learning_rate': 0.04, 'max_depth': 13, 'subsample': 1.0, 'colsample_bytree': 0.6, 'max_bin': 1194, 'min_child_weight': 8, 'gamm

In [12]:
best_params = study.best_params
print(f'Best Trial Params: {best_params}')

print(f'Best Trial Value: {study.best_trial.value}')

Best Trial Params: {'learning_rate': 0.04, 'max_depth': 10, 'subsample': 0.8, 'colsample_bytree': 1.0, 'max_bin': 2035, 'min_child_weight': 3, 'gamma': 0.05289953469172881, 'lambda': 2.1016544126433634, 'alpha': 5.086883026371227, 'grow_policy': 'lossguide'}
Best Trial Value: 0.003677589569155143


In [13]:
# optuna.visualization.plot_optimization_history(study)

In [14]:
# optuna.visualization.plot_slice(study)

In [15]:
# optuna.visualization.plot_param_importances(study)

# Submission

In [16]:
best_model = xgb.XGBRegressor(params=best_params)
best_model.fit(X, y)

X_test = df_test.drop(columns=['id'])
X_test = pd.get_dummies(X_test, columns=['Sex'], drop_first=True)
y_test_pred = best_model.predict(X_test)
y_test_pred = np.maximum(y_test_pred, 0)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,26.481358
1,750001,106.400833
2,750002,85.598892
3,750003,126.845436
4,750004,75.677055


#